## Main processing of the data for exploring word embeddings in the context of Ukrainian-English translation

### Installation

<i>Libraries to install with functions imports</i>

In [21]:
import fasttext
import numpy as np
import pandas as pd

from alignment import (
    learn_least_squares,
    build_candidate_matrices,
    translate_word,
    evaluate
)

from other_approaches import learn_orthogonal
from similarity_metrics import candidates_neighbours_evaluation_csls

[FastText](https://fasttext.cc/docs/en/crawl-vectors.html) embedding models for Ukrainian and English

In [22]:
model_uk = fasttext.load_model('../model/cc.uk.300.bin')
model_en = fasttext.load_model('../model/cc.en.300.bin')

### Data preparation

Creating pairs of words - Ukrainian and English

In [23]:
def data_to_dictionary(filename: str)-> dict:
    """
    Function creates from the data dictionary for 
    further usage

    :param filename: str, path to the data
    :return: dict, created dictionary
    """
    dict_res = {}
    with open(filename, "r", encoding="utf-8") as f_train:
        next(f_train)
        for line in f_train:
            line = line.strip("\n")
            ua_word, eng_word = line.split(" ")

            if ua_word not in dict_res:
                dict_res.setdefault(ua_word, [])

            dict_res[ua_word].append(eng_word)
    
    return dict_res

In [24]:
train_dict = data_to_dictionary("../data/usage/uk-en-train.csv")
test_dict = data_to_dictionary("../data/usage/uk-en-test.csv")
full_dictionary = data_to_dictionary("../data/usage/uk-en-full.csv")

In [25]:
pairs_train = []

for ua, eng_words in train_dict.items():
    for eng in eng_words:
        pairs_train.append((ua, eng))

### Main learning

Building aligned embeddings matrices for Ukrainian and English words

In [26]:
ua_embeddings, eng_embeddings = [], []

for ua, eng in pairs_train:
    ua_embeddings.append(model_uk.get_word_vector(ua))
    eng_embeddings.append(model_en.get_word_vector(eng))

A_ua, B_eng = np.column_stack(ua_embeddings), np.column_stack(eng_embeddings)

Learning the Least Squares Mapping

In [27]:
W_ls = learn_least_squares(A_ua, B_eng)

Preparing English Candidate words for translation

In [28]:
candidate_words = []
for _, eng_words in full_dictionary.items():
    for eng in eng_words:
        candidate_words.append(eng)

candidate_words, eng_candidate = build_candidate_matrices(candidate_words, model_en)
print("Number of candidate words:", len(candidate_words))

Number of candidate words: 28220


### Testing

Simple translation of a Ukrainian word with 3 different similarity metrics

In [29]:
translate_word("чай", W_ls, model_uk, candidate_words, eng_candidate, None, 5, "cosine")

[('tea', 0.6373974084854126),
 ('drink', 0.5615556836128235),
 ('coffee', 0.5396255254745483),
 ('wine', 0.5387119650840759),
 ('drinks', 0.529367983341217)]

In [31]:
r_y = candidates_neighbours_evaluation_csls(eng_candidate)
translate_word("чай", W_ls, model_uk, candidate_words, eng_candidate, r_y, 5, "csls")

[('tea', 0.20053523778915405),
 ('drink', -0.07791727781295776),
 ('coffee', -0.07868152856826782),
 ('beverage', -0.09232723712921143),
 ('beverages', -0.11303728818893433)]

In [30]:
translate_word("чай", W_ls, model_uk, candidate_words, eng_candidate, None, 5, "euclidean")

[('tea', 0.8515898585319519),
 ('drink', 0.9364233613014221),
 ('coffee', 0.959556519985199),
 ('wine', 0.9605082273483276),
 ('drinks', 0.9701876640319824)]

### Compare the main realisation with other approach
In this block, we compare 2 different approaches for aligning Ukrainian 
and English word embeddings. Each method uses its own transformation matrix W, 
but all methods are evaluated on the same test pairs and the same English candidate words. 

In [32]:
W_orthogonal = learn_orthogonal(A_ua, B_eng)

<i>Example of Ukrainian-English pairs of words</i>

In [33]:
print(list(test_dict.items())[:5])

[('зникли', ['gone']), ('петров', ['petrov']), ('банки', ['banks', 'jars', 'cans']), ('олія', ['oil']), ('менші', ['smaller'])]


Evaluate and compare the methods

In [35]:
methods = {
    "Least squares": W_ls,
    "Orthogonal Procrustes": W_orthogonal
}
comparison_results = []

for method_name, W in methods.items():
    evaluation = evaluate(test_dict, W, model_uk, 
                        candidate_words, eng_candidate, None, top_k=5)

    comparison_results.append({
        "method": method_name,
        "Precision@1": evaluation["Precision@1"],
        "Precision@5": evaluation["Precision@5"],
        "MRR": evaluation["MRR"],
        "Mean Rank": evaluation["Mean Rank"],
    })

comparison_df = pd.DataFrame(comparison_results)
comparison_df

,method,Precision@1,Precision@5,MRR,Mean Rank
0,Least squares,0.429272,0.605742,0.499475,3.289216
1,Orthogonal Procrustes,0.460784,0.621148,0.524230,3.185574


The **no-mapping** baseline gives 0% accuracy, which confirms that Ukrainian and English FastText embeddings are not directly comparable without alignment. **Least squares** and **ridge regression** both achieve 42.93% Top-1 accuracy and 60.57% Top-5 accuracy. The best method is **Orthogonal Procrustes**, with 46.08% Top-1 accuracy and 62.11% Top-5 accuracy. This suggests that **preserving the geometric structure** of the embedding space is more useful than learning an unrestricted linear mapping for this task.

In [36]:
r_y_csls = candidates_neighbours_evaluation_csls(eng_candidate)

similarity_metrics = ["cosine", "euclidean", "csls"]
ls_results = []

for metric in similarity_metrics:
    evaluation = evaluate(test_dict, W_ls, model_uk, candidate_words, eng_candidate,
                          r_y=r_y_csls if metric == "csls" else None, top_k=5,
                          similarity_metric=metric,
    )

    ls_results.append({
        "method": "Least Squares",
        "similarity_metric": metric,
        "Precision@1": evaluation["Precision@1"],
        "Precision@5": evaluation["Precision@5"],
        "MRR": evaluation["MRR"],
        "Mean Rank": evaluation["Mean Rank"],
    })

ls_df = pd.DataFrame(ls_results)
ls_df

,method,similarity_metric,Precision@1,Precision@5,MRR,Mean Rank
0,Least Squares,cosine,0.429272,0.605742,0.499475,3.289216
1,Least Squares,euclidean,0.429272,0.605742,0.499475,3.289216
2,Least Squares,csls,0.453081,0.620448,0.520261,3.191877
